# LeetCode #1278: Palindrome Partitioning III

https://leetcode.com/problems/palindrome-partitioning-iii/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(2^n \cdot n)$ | $O(n)$ |
| **Optimal: 2D DP ★** | $O(n^2 k)$ | $O(n^2)$ |

---

## Understanding the Methods

### Brute Force
Try all ways to partition the string into $k$ parts, computing the palindrome cost of each part. Exponential partitions make this infeasible for $n > 20$.

### Optimal: 2D DP ★
Precompute `cost[i][j]` = minimum insertions (equivalently, deletions from the reverse) to make `s[i..j]` a palindrome in $O(n^2)$. Then `dp[i][j]` = minimum changes to partition `s[0..i-1]` into `j` palindromes, transitioning over all last-chunk boundaries. Final answer: `dp[n][k]`.

**Constraints:**
* `1 <= s.length <= 100`
* `1 <= k <= s.length`
* `s` contains only lowercase English letters.

## Solutions

### C#

In [ ]:
public class Solution {
    public int PalindromePartition(string s, int k) {
        int n = s.Length;
        // cost[i][j] = min character changes to make s[i..j] a palindrome
        int[,] cost = new int[n, n];
        for (int len = 2; len <= n; len++) {
            for (int i = 0; i <= n - len; i++) {
                int j = i + len - 1;
                // Matching endpoints contribute 0, mismatched contribute 1
                cost[i, j] = (s[i] != s[j] ? 1 : 0) + (len > 2 ? cost[i + 1, j - 1] : 0);
            }
        }

        // dp[i][j] = min changes to split s[0..i-1] into j palindromes
        int[,] dp = new int[n + 1, k + 1];
        for (int i = 0; i <= n; i++)
            for (int j = 0; j <= k; j++)
                dp[i, j] = int.MaxValue / 2;
        dp[0, 0] = 0;

        for (int j = 1; j <= k; j++) {
            for (int i = j; i <= n; i++) {
                // Try every starting position of the last palindrome chunk
                for (int m = j - 1; m < i; m++) {
                    if (dp[m, j - 1] < int.MaxValue / 2)
                        dp[i, j] = Math.Min(dp[i, j], dp[m, j - 1] + cost[m, i - 1]);
                }
            }
        }
        return dp[n, k];
    }
}

### Python

In [ ]:
class Solution:
    def palindromePartition(self, s: str, k: int) -> int:
        n = len(s)
        # cost[i][j] = min character changes to make s[i..j] a palindrome
        cost = [[0] * n for _ in range(n)]
        for length in range(2, n + 1):
            for i in range(n - length + 1):
                j = i + length - 1
                cost[i][j] = (0 if s[i] == s[j] else 1) + (cost[i+1][j-1] if length > 2 else 0)

        # dp[i][j] = min changes to partition s[:i] into j palindromes
        INF = float('inf')
        dp = [[INF] * (k + 1) for _ in range(n + 1)]
        dp[0][0] = 0

        for j in range(1, k + 1):
            for i in range(j, n + 1):
                for m in range(j - 1, i):
                    if dp[m][j - 1] < INF:
                        dp[i][j] = min(dp[i][j], dp[m][j-1] + cost[m][i-1])

        return dp[n][k]

### Go

In [ ]:
func palindromePartition(s string, k int) int {
    n := len(s)
    // cost[i][j] = min changes to make s[i..j] a palindrome
    cost := make([][]int, n)
    for i := range cost { cost[i] = make([]int, n) }
    for length := 2; length <= n; length++ {
        for i := 0; i <= n-length; i++ {
            j := i + length - 1
            extra := 0
            if length > 2 { extra = cost[i+1][j-1] }
            if s[i] != s[j] { extra++ }
            cost[i][j] = extra
        }
    }

    const INF = 1<<30
    dp := make([][]int, n+1)
    for i := range dp {
        dp[i] = make([]int, k+1)
        for j := range dp[i] { dp[i][j] = INF }
    }
    dp[0][0] = 0

    for j := 1; j <= k; j++ {
        for i := j; i <= n; i++ {
            for m := j - 1; m < i; m++ {
                if dp[m][j-1] < INF {
                    if v := dp[m][j-1] + cost[m][i-1]; v < dp[i][j] {
                        dp[i][j] = v
                    }
                }
            }
        }
    }
    return dp[n][k]
}

### Rust

In [ ]:
impl Solution {
    pub fn palindrome_partition(s: String, k: i32) -> i32 {
        let s: Vec<u8> = s.bytes().collect();
        let n = s.len();
        let k = k as usize;
        // cost[i][j] = min changes to make s[i..=j] a palindrome
        let mut cost = vec![vec![0i32; n]; n];
        for length in 2..=n {
            for i in 0..=(n - length) {
                let j = i + length - 1;
                let extra = if length > 2 { cost[i+1][j-1] } else { 0 };
                cost[i][j] = extra + if s[i] == s[j] { 0 } else { 1 };
            }
        }

        let inf = i32::MAX / 2;
        let mut dp = vec![vec![inf; k + 1]; n + 1];
        dp[0][0] = 0;

        for j in 1..=k {
            for i in j..=n {
                for m in (j-1)..i {
                    if dp[m][j-1] < inf {
                        dp[i][j] = dp[i][j].min(dp[m][j-1] + cost[m][i-1]);
                    }
                }
            }
        }
        dp[n][k]
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `s = "abc", k = 2`
`cost[0][1]="ab"=1, cost[1][2]="bc"=1, cost[0][2]="abc"=1`. Best 2-partition: `"a"|"bc"` costs 0+1=1. Answer: **1**.

### 2. Slightly Complex
**Input:** `s = "aabbc", k = 3`
Partition `"aa"|"bb"|"c"` costs 0+0+0=0. Answer: **0**.

### 3. Edge Case: Time Factor
**Input:** `s` of length 100, `k = 50`.
Precomputing `cost` takes $O(n^2) = 10000$ operations; filling `dp` takes $O(n^2 k) = 500000$ — all feasible.

### 4. Edge Case: Space Factor
**Input:** `s` of length 100, `k = 100`.
`cost` and `dp` each hold $100 \times 100 = 10000$ integers — $O(n^2)$ space total.

### 5. Almost-Impossible but Plausible
**Input:** `s = "zzzzz...z"` (100 z's), `k = 1`.
The string is already a palindrome; `cost[0][99] = 0`. Answer: **0** with zero modifications.